# t-Test — Theory, Math, and a Worked Example (with Executable Python)

If your viewer supports MathJax/LaTeX (e.g., **JupyterLab**, **classic Jupyter Notebook**, **Google Colab**), the equations below will render properly.

**Tip (very important):** Some environments (notably many **VS Code notebook previews** or markdown renderers without MathJax) will *not* render LaTeX math. In those cases, open this notebook in **Google Colab** or a full Jupyter environment.

---

## What is a t-test?

A **t-test** is a statistical method used to test whether an observed difference in means is likely to be **real** or could plausibly be explained by **random variation** (“noise”).

It’s especially useful when:

- Sample size is relatively small, and/or  
- The **population variance is unknown** (almost always true in practice).

---

## Core intuition: signal-to-noise ratio

A t-test is essentially:

> **(difference you care about)** ÷ **(how noisy your measurements are)**

If the difference is large compared to the noise, the test statistic has a large magnitude and becomes unlikely under the “no-difference” assumption.

---

## One-sample t-test (math)

We test whether a sample mean differs from a reference value $\\mu_0$.

### Test statistic

$$
t = \\frac{\\bar{x} - \\mu_0}{s/\\sqrt{n}}
$$

Where:
- $\\bar{x}$ = sample mean  
- $\\mu_0$ = reference mean under the null hypothesis  
- $s$ = sample standard deviation (estimated from the sample; uses $n-1$ in the variance)  
- $n$ = sample size  

### Degrees of freedom

$$
df = n - 1
$$

---

## Hypotheses (the “question” inside a t-test)

Two-sided one-sample test:

- $H_0: \\mu = \\mu_0$  (no difference)  
- $H_1: \\mu \\ne \\mu_0$ (difference exists)  

---

## Why t (not z)?

If the population standard deviation $\\sigma$ were known, we'd use the z-score:

$$
z = \\frac{\\bar{x} - \\mu_0}{\\sigma/\\sqrt{n}}
$$

But we replace $\\sigma$ with an estimate $s$, which introduces extra uncertainty. That’s why the statistic follows a **t-distribution** rather than a normal distribution.

As $n$ grows, the t-distribution approaches the normal distribution.


## Worked example (one-sample t-test)

Suppose you trained a model 5 times and got accuracies:

$$
x = [78, 82, 80, 85, 79]
$$

Test whether the mean accuracy differs from $75$:

$$
H_0: \\mu = 75
\\qquad
H_1: \\mu \\ne 75
$$

We will compute:

- sample mean $\\bar{x}$  
- sample standard deviation $s$  
- t-statistic $t$  
- degrees of freedom $df$  
- p-value (two-sided)  
- and (optionally) the critical value at $\\alpha=0.05$


In [1]:
import math, statistics

# Data for the worked example
x = [78, 82, 80, 85, 79]
mu0 = 75  # null/reference mean

n = len(x)
xbar = sum(x) / n

# Unbiased sample standard deviation (denominator n-1)
s = statistics.stdev(x)

t_stat = (xbar - mu0) / (s / math.sqrt(n))
df = n - 1

print("n =", n)
print("mean (x̄) =", xbar)
print("stdev (s) =", s)
print("t-statistic =", t_stat)
print("df =", df)


n = 5
mean (x̄) = 80.8
stdev (s) = 2.7748873851023217
t-statistic = 4.673773191347203
df = 4


## p-value (two-sided)

The p-value is:

> Probability (under $H_0$) of seeing a t-statistic at least as extreme as the observed one.

Formally (two-sided):

$$
p = 2\\,\\Bigl(1 - F_{t,df}(|t|)\\Bigr)
$$

Where $F_{t,df}$ is the CDF of the t-distribution with $df$ degrees of freedom.


In [2]:
# Compute a two-sided p-value.
# We'll use SciPy if available (recommended). If SciPy isn't available, we use a fallback normal approximation.

p_value = None
used = None

try:
    from scipy import stats
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))
    used = "scipy.stats.t"
except Exception as e:
    used = f"fallback (SciPy not available: {e})"
    # NOTE: This is only a rough approximation. Prefer SciPy, especially for small df.
    def normal_cdf(z):
        return 0.5 * (1 + math.erf(z / math.sqrt(2)))
    p_value = 2 * (1 - normal_cdf(abs(t_stat)))

print("p-value =", p_value)
print("method =", used)


p-value = 0.009491601789400672
method = scipy.stats.t


## Decision rule at significance level $\\alpha = 0.05$

- If $p < \\alpha$, **reject** $H_0$
- If $p \\ge \\alpha$, **fail to reject** $H_0$

You can also compare against a critical value $t_{crit}$:

$$
|t| > t_{crit} \\Rightarrow \\text{reject } H_0
$$

For a two-sided test at $\\alpha=0.05$, we use the $97.5\\%$ percentile of the t-distribution:

$$
t_{crit} = \\mathrm{t\_ppf}(1-\\alpha/2, df)
$$


In [3]:
alpha = 0.05
t_crit = None

try:
    from scipy import stats
    t_crit = stats.t.ppf(1 - alpha/2, df=df)
except Exception as e:
    print("SciPy not available, cannot compute t critical value accurately:", e)

decision = "REJECT H0" if p_value < alpha else "FAIL TO REJECT H0"

print("alpha =", alpha)
print("t_crit =", t_crit)
print("decision =", decision)


alpha = 0.05
t_crit = 2.7764451051977987
decision = REJECT H0


## Summary

A t-test is a hypothesis test framed as:

- $H_0$ (null): no effect / no difference  
- $H_1$ (alternative): effect / difference  

The t-statistic measures how large the observed difference is compared to the sample noise level.

---

## Quick utility function (one-sample t-test)

Returns $t$, $df$, and a two-sided p-value.


In [4]:
def one_sample_t_test(sample, mu0=0.0):
    import math, statistics
    n = len(sample)
    xbar = sum(sample) / n
    s = statistics.stdev(sample)
    t = (xbar - mu0) / (s / math.sqrt(n))
    df = n - 1
    try:
        from scipy import stats
        p = 2 * (1 - stats.t.cdf(abs(t), df=df))
        method = "scipy"
    except Exception:
        # rough normal approximation fallback
        p = 2 * (1 - (0.5 * (1 + math.erf(abs(t) / math.sqrt(2)))))
        method = "normal-approx (rough)"
    return {"t": t, "df": df, "p_value_two_sided": p, "mean": xbar, "stdev": s, "n": n, "method": method}

one_sample_t_test(x, mu0=75)


{'t': 4.673773191347203,
 'df': 4,
 'p_value_two_sided': np.float64(0.009491601789400672),
 'mean': 80.8,
 'stdev': 2.7748873851023217,
 'n': 5,
 'method': 'scipy'}